# WignerFlow: full argon retrain (October 29)

Run all cells in a Colab **GPU** runtime. This notebook clones `feat/colab-oct29`, installs the package, generates ten 0.5 ns OpenMM CUDA trajectories, trains the original `configs/argon/B_S3a_seed1.yaml` curriculum for 40,000 steps, exports the final checkpoint, and evaluates it against v0 over 8 starts × 3 seeds × 1,000 steps. Model-only dynamics, 10 flow steps, no thermostat.

Everything persistent is under `/content/drive/MyDrive/WignerFlow_runs/argon_full_oct29`. Keep this path to resume; change `RUN_NAME` for an independent experiment. Dataset completion markers skip finished trajectories; an interrupted trajectory is regenerated. Training resumes automatically from `train/ckpt.pt`, including optimizer/step/best score. The current trainer does not save RNG/DataLoader state, so continuation is not bit-for-bit reproducible.

**Runtime estimate (not a GPU measurement):** budget roughly 8–36 hours on T4, 4–20 hours on L4, or 2–12 hours on A100, plus data generation and evaluation. Colab sessions may expire before completion; rerun all cells to resume. The CPU measurements in `docs/RETRAIN_NOTES_oct29.md` give 1.76 s/update at batch 1, m=1, and 2.24 s/update at m=2, versus the requested batch 8 and 40,000 updates. A crude linear batch extrapolation is 156–200 CPU hours before m=4 and validation; assuming 5–80× GPU acceleration gives a broad multi-hour to multi-day range. This assumption is unverified. The calibration cell below measures this GPU and saves an improved projection for the actual curriculum, validation and final evaluation. Data generation was 16.73 CPU seconds for four 20 ps productions plus four 20 ps equilibrations; scaling simulated duration to ten 500 ps + 20 ps is about 544 CPU seconds, excluding the much larger disk/Drive I/O. Allow tens of minutes for storage. Full data/caches require several GB (allow ≥20 GB free locally and on Drive).

The physical settings were reconstructed, not recovered byte for byte; see `docs/RETRAIN_NOTES_oct29.md`. The production count/duration restore the original budget. This notebook has been parsed and its Python cells compiled locally; CUDA execution must be done in Colab.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, time, shutil

drive.mount('/content/drive')
RUN_NAME = 'argon_full_oct29'
RUN = Path('/content/drive/MyDrive/WignerFlow_runs') / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/WignerFlow')
BRANCH = 'feat/colab-oct29'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENMM_CPU_THREADS'] = '2'
os.environ.pop('PYTHONPATH', None)

def run(command, log_name=None):
    print(' '.join(map(str, command)), flush=True)
    start = time.perf_counter()
    log = open(RUN / log_name, 'a') if log_name else None
    try:
        with subprocess.Popen(list(map(str, command)), cwd=REPO if REPO.exists() else '/content',
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='')
                if log:
                    log.write(line)
                    log.flush()
            status = proc.wait()
        with open(RUN / 'commands.jsonl', 'a') as f:
            f.write(json.dumps(dict(command=list(map(str, command)), wall_s=time.perf_counter()-start, exit=status))+'\n')
        if status:
            raise RuntimeError(f'Command failed with exit {status}')
    finally:
        if log:
            log.close()


In [ ]:
if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/yaniguan/WignerFlow.git', REPO])
run(['git', 'checkout', BRANCH])
# Existing clones stay pinned on rerun. To update, deliberately pull before training.
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-e', '.[simulation,train,test]'], 'install.log')
run([sys.executable, '-m', 'pip', 'freeze'], 'environment.txt')
run(['git', 'rev-parse', 'HEAD'], 'source_commit.txt')
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then rerun.'
torch.set_num_threads(2)
print(torch.cuda.get_device_name(0))
(RUN / 'gpu.json').write_text(json.dumps(dict(device=torch.cuda.get_device_name(0), torch=torch.__version__), indent=2))
run([sys.executable, '-m', 'pytest', '-q'], 'pytest.log')


## Generate the full dataset and stage it locally

HDF5 files go directly to Drive, so completed trajectories survive a runtime reset. Never point this full-data command at the old reduced `data/argon` directory: the generator skips already-complete files without comparing their production duration. Local copies and local memory-mapped caches speed up training. The original training YAML remains intact; only data/output locations and checkpoint frequency are changed in a saved runtime copy.


In [ ]:
import yaml
DATA_DRIVE = RUN / 'data' / 'argon_full'
DATA_LOCAL = REPO / 'data' / 'argon_full'
DATA_DRIVE.mkdir(parents=True, exist_ok=True)
DATA_LOCAL.mkdir(parents=True, exist_ok=True)
data_cfg = yaml.safe_load((REPO / 'configs/data/argon_full.yaml').read_text())
assert data_cfg['n_traj'] == 10 and data_cfg['production_ps'] == 500.0
run([sys.executable, 'scripts/gen_data.py', '--config', 'configs/data/argon_full.yaml',
     '--platform', 'CUDA', '--out_dir', DATA_DRIVE], 'generation.log')
import h5py
for i in range(10):
    src = DATA_DRIVE / f'traj_{i:03d}.h5'
    with h5py.File(src, 'r') as h:
        assert h.attrs['complete'] and len(h['positions']) == 100000, str(src)
    dst = DATA_LOCAL / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
shutil.copy2(DATA_DRIVE / 'config.yaml', DATA_LOCAL / 'config.yaml')
cfg = yaml.safe_load((REPO / 'configs/argon/B_S3a_seed1.yaml').read_text())
assert cfg['train']['steps'] == 40000
cfg['data_dir'] = str(DATA_LOCAL)
cfg['out_dir'] = str(RUN / 'train')
cfg['train']['ckpt_minutes'] = 10
CONFIG = RUN / 'B_S3a_seed1_runtime.yaml'
CONFIG.write_text(yaml.safe_dump(cfg))
shutil.copy2(REPO / 'configs/argon/B_S3a_seed1.yaml', RUN / 'B_S3a_seed1_original.yaml')
shutil.copy2(REPO / 'configs/data/argon_full.yaml', RUN / 'argon_full_source.yaml')
print('Runtime config:', CONFIG)


## Calibrate this GPU (optional on a resumed session)

This uses a separate temporary model; it never changes training checkpoints. It measures batch 8 at each unroll length, plus a 10-step rollout at validation batch size 4. Estimated totals include configured validation rollouts and 48,000 single-rollout steps for the final two-checkpoint evaluation, but exclude ground-truth analysis, validation loss, startup and I/O. Allow substantial overhead. If the original batch 8 exhausts GPU memory, select a larger GPU to retain the requested training recipe.


In [ ]:
from equitraj.dataset import TrajWindows
from equitraj.model import EquiTraj
from equitraj.train import training_loss
from equitraj.rollout import rollout, temperature
import numpy as np

CALIBRATION = RUN / 'gpu_calibration.json'
if not CALIBRATION.exists():
    ds = TrajWindows(DATA_LOCAL, 'train', cfg['stride_steps'], cfg['k'], m=4, start_every=100)
    items = [ds[i] for i in range(cfg['train']['batch_size'])]
    pos = torch.stack([i[0] for i in items]).cuda()
    vel = torch.stack([i[1] for i in items]).cuda()
    box = torch.stack([i[2] for i in items]).cuda()
    types, masses = ds.types.cuda(), ds.masses.cuda()
    torch.manual_seed(1)
    model = EquiTraj(len(ds.elements), k=cfg['k'], stride_fs=ds.stride_fs, **cfg['model']).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['train']['lr'])
    timings = {}
    for m in [1, 2, 4]:
        for j in range(4):
            if j == 1:
                torch.cuda.synchronize()
                t0 = time.perf_counter()
            opt.zero_grad(set_to_none=True)
            loss = training_loss(model, pos, vel, box, types, masses, m, cfg['train']['noise_sigma'],
                                 cfg['train']['backprop'], cfg['train']['unroll_sampling_steps'])
            loss.backward()
            opt.step()
        torch.cuda.synchronize()
        timings[f'm{m}_s_per_update'] = (time.perf_counter()-t0)/3
    model.eval()
    for batch in [1, 4]:
        out = rollout(model, pos[:batch, :cfg['k']], vel[:batch, :cfg['k']], ds.types, box[:batch], ds.masses,
                      10, sampling_steps=10)
        timings[f'rollout_batch{batch}_s_per_step'] = out['wall_s']/out['steps_done']
    train_s = 10000*timings['m1_s_per_update'] + 10000*timings['m2_s_per_update'] + 20000*timings['m4_s_per_update']
    timings['projected_train_hours_excluding_validation_loss_IO'] = (train_s + 20*1000*timings['rollout_batch4_s_per_step'])/3600
    timings['projected_final_eval_hours_excluding_analysis_IO'] = 2*8*3*1000*timings['rollout_batch1_s_per_step']/3600
    CALIBRATION.write_text(json.dumps(timings, indent=2))
    del model, opt, ds, items, pos, vel, box, types, masses, loss, out
    import gc
    gc.collect()
    torch.cuda.empty_cache()
print(CALIBRATION.read_text())


## Train and resume

Checkpoints are written to Drive every 10 minutes and every 2,000-step validation, with atomic `ckpt.pt.tmp` → `ckpt.pt` replacement. `best.pt` tracks validation selection; every validation also creates a snapshot. Rerun this cell (or all cells after a reset) to continue from the most recent `ckpt.pt`. The final export uses the prespecified step-40,000 snapshot, without choosing a checkpoint using test results.


In [ ]:
run([sys.executable, '-m', 'equitraj.train', '--config', CONFIG, '--device', 'cuda'], 'training.log')
done = json.loads((RUN / 'train/done.json').read_text())
assert done['step'] == 40000
EXPORT = RUN / 'checkpoints' / 'argon_flow_full_step40000.pt'
EXPORT.parent.mkdir(exist_ok=True)
run([sys.executable, 'scripts/export_ckpt.py', '--run', RUN / 'train',
     '--ckpt', 'snapshots/step040000.pt', '--out', EXPORT], 'export.log')
BASELINE = EXPORT.parent / 'argon_flow_v0.pt'
shutil.copy2(REPO / 'checkpoints/argon_flow_v0.pt', BASELINE)
shutil.copy2(REPO / 'checkpoints/argon_flow_v0.pt.sha256', str(BASELINE)+'.sha256')


## Paired multistart / multiseed comparison

Both checkpoints use the same held-out full-data trajectory, evenly spaced starting windows and explicit seeds 0, 1, 2. CIs resample starts while retaining their seeds together. Physical metrics condition on surviving all 1,000 steps; missing VACF zeros remain null and their counts are reported. Eight starts from one trajectory can overlap and do not measure training-seed or independent-trajectory uncertainty. A 50 ps rollout is shorter than the original 0.5 ns release benchmark.


In [ ]:
EVAL = RUN / 'eval_multiseed'
run([sys.executable, 'scripts/eval_multiseed.py', '--checkpoints', BASELINE, EXPORT,
     '--data_dir', DATA_LOCAL, '--starts', '8', '--seeds', '3', '--steps', '1000',
     '--sampling_steps', '10', '--device', 'cuda', '--threads', '2', '--out', EVAL], 'evaluation.log')
from IPython.display import display, Markdown
display(Markdown((EVAL / 'metrics.md').read_text()))


## Archive and download

The zip includes training checkpoints, exported weights, metrics, configs, timings, and logs. The large full dataset remains separately on Drive under `data/argon_full`; local `.npy` caches can be regenerated. The archive is also saved to Drive. Browser download may be large because validation snapshots and optimizer state are retained.


In [ ]:
import zipfile
from google.colab import files
ARCHIVE = RUN.parent / (RUN_NAME + '_results.zip')
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in sorted(RUN.rglob('*')):
        if path.is_file() and 'data' not in path.relative_to(RUN).parts and not path.name.endswith('.tmp'):
            z.write(path, arcname=str(Path(RUN_NAME) / path.relative_to(RUN)))
print('Results:', RUN)
print('Full HDF5 data:', DATA_DRIVE)
print('Archive:', ARCHIVE, ARCHIVE.stat().st_size, 'bytes')
files.download(str(ARCHIVE))
